In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm
from matplotlib.colors import SymLogNorm
import yt
import os
from dotenv import dotenv_values
import glob
import pandas as pd


In [ ]:
# raw_dir = f"snapshots_Nx{Nx}_Ny{Ny}_seed{SEED}_mgs{MAX_GRID_SIZE}_dt{FIXED_DT}_visc_coef{VISC_COEF}_noise_off_step{NOISE_OFF_STEP}_depth{CELL_DEPTH}"
# run_dir = raw_dir.replace('.', 'p')
# # plt_dir = os.path.join(run_dir, "plt")

# run_dir = ""
run_dir = "SF_32_1em2_3em9_seed_1_fft"
plt_dir = os.path.join(run_dir, "plt_SF_mag0*")
plotfile_paths = sorted(glob.glob(plt_dir))

print(f"Found {len(plotfile_paths)} plotfiles in {run_dir}.")

ts = yt.DatasetSeries(plotfile_paths)

first_ds = ts[0]
print("\nAvailable variables in the plotfiles:")
for field in first_ds.field_list:
    print(f" - {field[1]}") 

In [ ]:
run_dir_2 = "SF_32_1em2_3em9_seed_2_fft"
plt_dir_2 = os.path.join(run_dir_2, "plt_SF_mag0*")
plotfile_paths_2 = sorted(glob.glob(plt_dir_2))
ts_2 = yt.DatasetSeries(plotfile_paths_2)
print(f"Found {len(plotfile_paths_2)} plotfiles in {plt_dir_2}.")

run_dir_3 = "SF_32_1em2_3em9_seed_3_fft"
plt_dir_3 = os.path.join(run_dir_3, "plt_SF_mag0*")
plotfile_paths_3 = sorted(glob.glob(plt_dir_3))
ts_3 = yt.DatasetSeries(plotfile_paths_3)
print(f"Found {len(plotfile_paths_3)} plotfiles in {plt_dir_3}.")

In [ ]:
def structure_factor_helper(ax, data, title, cmap='RdBu', fontsize=13):
    """Plot structure factor data with colorbar on top in separated axes"""
    plot_height, plot_bottom = 0.75, 0.05
    cbar_height, cbar_bottom = 0.04, 0.85
    plot_width, plot_left = 0.90, 0.075  
    
    pbbox = transforms.Bbox.from_bounds(plot_left, plot_bottom, plot_width, plot_height)
    cbbox = transforms.Bbox.from_bounds(plot_left, cbar_bottom, plot_width, cbar_height)
    
    to_axes_bbox = transforms.BboxTransformTo(ax.get_position())
    paxes = ax.figure.add_axes(pbbox.transformed(to_axes_bbox))
    caxes = ax.figure.add_axes(cbbox.transformed(to_axes_bbox))
    ax.axis('off')
    
    Ny, Nx = data.shape
    extent = [-Nx/2, Nx/2, -Ny/2, Ny/2]  
    
    # vmax = np.max(np.abs(data))
    # im = paxes.imshow(data, cmap=cmap, origin='lower', extent=extent, aspect=1, 
    #                   norm=SymLogNorm(linthresh=1e-15, vmin=-vmax, vmax=vmax, base=10))

    vmax = data.max()
    # vmax = 3e-13
    # vmin = 1e-16
    # vmin = 1e-4
    # im = paxes.imshow(data, cmap=cmap, origin='lower', extent=extent, aspect=1, norm=LogNorm(vmin=vmin, vmax=vmax))
    im = paxes.imshow(data, cmap=cmap, origin='lower', extent=extent, aspect=1, vmin=0, vmax=vmax)
    
    paxes.set_xlabel('$k_x$', fontsize=fontsize)
    paxes.set_ylabel('$k_y$', fontsize=fontsize)
    paxes.tick_params(length=0, width=0, labelsize=fontsize-2)
    
    caxes.text(0.5, 3.1, title, transform=caxes.transAxes, 
               ha='center', va='bottom', fontsize=fontsize)
    
    cbar = plt.colorbar(im, cax=caxes, orientation='horizontal')
    cbar.outline.set_visible(False)
    caxes.xaxis.set_ticks_position('top')
    cbar.ax.tick_params(labelsize=fontsize-2)
    
    return paxes, caxes


yt.funcs.mylog.setLevel(40)
Nx, Ny = first_ds.domain_dimensions[:2]
nrows, ncols = 1, 2
# cmap = 'magma'
# cmap = 'Blues'
cmap = 'RdBu'
for i in range(95, len(ts)):
    ds = ts[i]
    cg = ds.covering_grid(level=0, left_edge=ds.domain_left_edge, dims=[Nx, Ny, 1])
    
    S_xx = np.array(cg['struct_fact_velx_velx'][:, :, 0].v)
    S_yy = np.array(cg['struct_fact_vely_vely'][:, :, 0].v)
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 6), dpi=150)
    fig.suptitle(f'Seed A Staggered Code Structure Factor | t = {ds.current_time.v:.5e}', fontsize=15, y=0.94)
    
    structure_factor_helper(axes[0], S_xx.T, r'$S_{v_x, v_x}(\mathbf{k})$', cmap = cmap)
    structure_factor_helper(axes[1], S_yy.T, r'$S_{v_y, v_y}(\mathbf{k})$', cmap = cmap)
    
    plt.show()

In [ ]:
def df_trace_f(ts):
    # Get domain dimensions
    Nx, Ny = ts[0].domain_dimensions[:2]
    
    # Initialize lists to store our data
    times = []
    traces = []
    
    # Loop over the time series
    for ds in ts:
        # Get the physical simulation time
        sim_time = ds.current_time.v
        
        # Extract the 2D arrays using covering_grid
        cg = ds.covering_grid(level=0, left_edge=ds.domain_left_edge, dims=[Nx, Ny, 1])
        S_xx = np.array(cg['struct_fact_velx_velx'][:, :, 0].v)
        S_yy = np.array(cg['struct_fact_vely_vely'][:, :, 0].v)
        
        # Calculate the trace (sum of diagonals)
        # We sum over the entire k-space array to get the total energy trace for this timestep
        # trace_val = np.sum(np.power(S_xx + S_yy, 1/2))
        trace_val = np.sum(S_xx + S_yy)
        
        # Append to lists
        times.append(sim_time)
        traces.append(trace_val)
    
    # Create the Pandas DataFrame
    df_trace = pd.DataFrame({
        'Time': times,
        'Trace': traces
    })

    return df_trace

# Display the first few rows
# print(df_trace)

df_trace_1 = df_trace_f(ts)
df_trace_2 = df_trace_f(ts_2)
df_trace_3 = df_trace_f(ts_3)

In [ ]:
kB = 1.380649e-16
T = 294.0
# cell_depth = 3.0e-9
# kB = 1.
# T = 1.0
# cell_depth = 1.
# L = 3200
L = 1
dx, dy = L/Nx, L/Ny
# dV = dx * dy * cell_depth
# print("dV: ", dV)
rho = 1

# dx1, dy1 = 1/Nx, 1/Ny
# dV1 = dx1 * dy1 * cell_depth

answer = (kB * T) / (rho)
# answer = (kB * T) / (rho * dV)
# answer = (kB * T * dV) / (rho)
print("answer: ", answer)
# print("np.power(Nx, 2): ", np.power(Nx, 2))
# print(df_trace['Trace'] / (np.power(Nx, 2) * dV1))
# print(df_trace['Trace'] / (np.power(Nx, 2)))

In [ ]:
plt.figure(dpi=300, figsize=(8, 6))
# plt.plot(df_trace['Time'], df_trace['Trace'] / np.square(Nx), c = "blue", label = r"$Tr(S_{u,u}) / N_x N_y$")
# plt.axhline(answer, label = r"$k_B \: T / \rho$", c = "black", ls = "--")
plt.plot(df_trace_1['Time'], (df_trace_1['Trace'] / np.square(Nx))/answer, c = "blue", label = r"$Tr(S_{u,u}) / N_x N_y$")
plt.plot(df_trace_2['Time'], (df_trace_2['Trace'] / np.square(Nx))/answer, c = "green", label = r"$Tr(S_{u,u}) / N_x N_y$")
plt.plot(df_trace_3['Time'], (df_trace_3['Trace'] / np.square(Nx))/answer, c = "red", label = r"$Tr(S_{u,u}) / N_x N_y$")
plt.axhline(answer/answer, label = r"$k_B \: T / \rho$", c = "black", ls = "--")
plt.xlabel(r"$t$")
plt.title(f"Normalized Trace for FFT Staggered Code with multiple seeds\n(N_x = {Nx}, dt = 3.9e-4, visc = 1e-2, cell_depth = 3e-9)")
plt.legend()
plt.show()
plt.close()

In [ ]:
# num_avg_steps = 10
# recent_ts = ts[-num_avg_steps:]

# # Initialize an empty 2D array to hold the averaged trace
# avg_trace_2d = np.zeros((Nx, Ny))

# for ds in recent_ts:
#     cg = ds.covering_grid(level=0, left_edge=ds.domain_left_edge, dims=[Nx, Ny, 1])
#     S_xx = np.array(cg['struct_fact_velx_velx'][:, :, 0].v)
#     S_yy = np.array(cg['struct_fact_vely_vely'][:, :, 0].v)
    
#     # Add the trace array for this timestep to the running total
#     avg_trace_2d += (S_xx + S_yy)

# # Divide by the number of steps to get the average
# avg_trace_2d /= num_avg_steps

ds_final = ts[-1]
# ds_final = ts[7]

# Extract the covering grid
cg = ds_final.covering_grid(level=0, left_edge=ds_final.domain_left_edge, dims=[Nx, Ny, 1])

# Extract S_xx and S_yy and sum them to get the Trace
S_xx = np.array(cg['struct_fact_velx_velx'][:, :, 0].v)
S_yy = np.array(cg['struct_fact_vely_vely'][:, :, 0].v)

trace_2d = S_xx + S_yy

# trace_2d = trace_2d / (Nx * Ny)**2

In [ ]:
# len(ts) - num_avg_steps
ts[30:]

In [ ]:
plt.figure(figsize=(8, 6), dpi=150)

# Set extent to match k-space (centered at 0)
extent = [-Nx/2, Nx/2, -Ny/2, Ny/2]
# vmax = trace_2d.max()
# vmin = 1e-2
# vmax = 6e-14
# vmax = 3e-13
vmin = .8 * answer
vmax = 1.2 * answer
# vmin = .2 * answer
# vmax = 1.8 * answer
im = plt.imshow(trace_2d.T, origin='lower', extent=extent, cmap='magma', vmin = vmin, vmax = vmax)
# im = plt.imshow(trace_2d.T, origin='lower', extent=extent, cmap='magma', 
#                 norm=LogNorm(vmin=vmin, vmax=vmax))

plt.colorbar(im, label=r'$\langle \text{Tr}(\mathbf{S}_{u,u}) \rangle$')
# start_time = recent_ts[0].current_time.v
# end_time = recent_ts[-1].current_time.v
time = ds_final.current_time.v

# plt.title(f'Averaged Trace of Structure Factor (Staggered code) \n(t = {start_time:.2e} to {end_time:.2e})')
plt.title(f'Trace of Structure Factor (FFT Staggered code)\n(Seed A, t = {time:.2e})')

# plt.title(f'Averaged Trace of Structure Factor (Last {num_avg_steps} steps)')
plt.xlabel('$k_x$')
plt.ylabel('$k_y$')
plt.show()